# Real or AI? A lightweight deepfake / AI-generated image detector
**EECE 490/690 - Introduction to Machine Learning - Project 18**

This notebook reproduces the whole project on Google Colab:

1. build the dataset (real MS COCO photos vs. images from five modern generators)
2. fine-tune a small pretrained CNN (**transfer learning**) with social-feed style **data augmentation**
3. evaluate accuracy, robustness and computational cost
4. launch the demo screen: place two images side by side and check which one is AI-generated

**Before running:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

## 1. Setup

In [ ]:
!pip -q install "gradio>=6.0"
import torch, torchvision
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none - switch the runtime to T4 GPU")

## 2. Project code
The project is three Python files. The next three cells write them to the Colab disk; they are identical to the
`.py` files in the submission.

| File | Role |
|---|---|
| `detector.py` | the model, the augmentations, prediction + Grad-CAM heat-map |
| `train.py` | dataset preparation, training, evaluation, comparison of runs |
| `app.py` | the two-image demo screen |

In [ ]:
%%writefile detector.py
"""
detector.py - the shared core of the project (used by train.py and app.py).

    DeepfakeDetector   light ImageNet-pretrained backbone + two small heads
                         fake head   : real photo (0) vs AI-generated (1)
                         source head : which generator the image most resembles
    train_transform    preprocessing + "social-feed" data augmentation
    eval_transform     preprocessing only (optionally with a fixed degradation)
    load_detector      rebuild a trained model from a checkpoint
    analyze            one image -> probability, likely generator, Grad-CAM heat-map
"""
import io
import random
from functools import partial

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageFilter
from torchvision import models
from torchvision import transforms as T

IMG_SIZE = 224
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]  # ImageNet statistics

# The five generators in the dataset: file-name prefix -> display name.
SOURCES = {"sd21": "Stable Diffusion 2.1", "sdxl": "SDXL", "sd3": "Stable Diffusion 3",
           "dalle3": "DALL-E 3", "midjourney": "Midjourney v6"}
SOURCE_KEYS = list(SOURCES)

# Light backbones: name -> (torchvision constructor, channels of the last feature map).
BACKBONES = {
    "mobilenet_v3_small": (models.mobilenet_v3_small, 576),
    "mobilenet_v3_large": (models.mobilenet_v3_large, 960),
    "mobilenet_v2": (models.mobilenet_v2, 1280),
    "efficientnet_b0": (models.efficientnet_b0, 1280),
}


# ----------------------------------------------------------------------------- model
class DeepfakeDetector(nn.Module):
    """Pretrained convolutional features -> global average pool -> two linear heads."""

    def __init__(self, backbone="mobilenet_v3_large", pretrained=True):
        super().__init__()
        build, channels = BACKBONES[backbone]
        self.backbone_name = backbone
        self.features = build(weights="DEFAULT" if pretrained else None).features
        self.dropout = nn.Dropout(0.2)
        self.fake_head = nn.Linear(channels, 1)
        self.source_head = nn.Linear(channels, len(SOURCES))

    def heads(self, feature_map):
        pooled = self.dropout(feature_map.mean(dim=(2, 3)))
        return self.fake_head(pooled).squeeze(1), self.source_head(pooled)

    def forward(self, x):
        """Returns (fake logit [B], generator logits [B, 5])."""
        return self.heads(self.features(x))

    def freeze_backbone(self, frozen=True):
        for p in self.features.parameters():
            p.requires_grad = not frozen


# ----------------------------------------------------------------------------- transforms
def degrade(img, scale=None, blur=None, jpeg=None):
    """What a social feed does to a picture: shrink it, soften it, re-compress it."""
    if scale:
        w, h = img.size
        small = img.resize((max(8, int(w * scale)), max(8, int(h * scale))), Image.BILINEAR)
        img = small.resize((w, h), Image.BICUBIC)
    if blur:
        img = img.filter(ImageFilter.GaussianBlur(blur))
    if jpeg:
        buffer = io.BytesIO()
        img.save(buffer, "JPEG", quality=int(jpeg))
        img = Image.open(io.BytesIO(buffer.getvalue())).convert("RGB")
    return img


class FeedAugment:
    """Random version of `degrade`, applied during training."""

    def __call__(self, img):
        return degrade(
            img,
            scale=random.uniform(0.4, 0.9) if random.random() < 0.3 else None,
            blur=random.uniform(0.3, 1.5) if random.random() < 0.2 else None,
            jpeg=random.randint(30, 95) if random.random() < 0.5 else None,
        )


def eval_transform(**degradation):
    """Resize + centre crop. Pass scale= / blur= / jpeg= to test robustness."""
    steps = [T.Resize(256), T.CenterCrop(IMG_SIZE)]
    if degradation:
        steps.append(partial(degrade, **degradation))
    return T.Compose(steps + [T.ToTensor(), T.Normalize(MEAN, STD)])


def train_transform(augment=True):
    if not augment:
        return eval_transform()
    return T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.3, 1.0), ratio=(3 / 4, 4 / 3)),
        T.RandomHorizontalFlip(),
        T.ColorJitter(0.2, 0.2, 0.2, 0.02),
        FeedAugment(),
        T.ToTensor(),
        T.Normalize(MEAN, STD),
    ])


# ----------------------------------------------------------------------------- inference
def load_detector(path="models/detector.pt", device="cpu"):
    checkpoint = torch.load(path, map_location=device, weights_only=True)
    model = DeepfakeDetector(checkpoint["backbone"], pretrained=False)
    model.load_state_dict(checkpoint["state_dict"])
    return model.to(device).eval()


def analyze(model, image, heatmap=True):
    """
    Classify one PIL image.

    Returns {"p_fake": probability the image is AI-generated,
             "sources": {generator name: probability},
             "heatmap": PIL image showing the regions behind the verdict (Grad-CAM)}.
    """
    device = next(model.parameters()).device
    crop = T.Compose([T.Resize(256), T.CenterCrop(IMG_SIZE)])(image.convert("RGB"))
    x = T.Normalize(MEAN, STD)(T.ToTensor()(crop)).unsqueeze(0).to(device)

    with torch.set_grad_enabled(heatmap):
        feature_map = model.features(x)
        fake_logit, source_logits = model.heads(feature_map)
    p_fake = torch.sigmoid(fake_logit).item()
    source_probs = torch.softmax(source_logits, dim=1)[0].tolist()
    result = {"p_fake": p_fake, "sources": dict(zip(SOURCES.values(), source_probs)), "heatmap": None}

    if heatmap:
        # Grad-CAM: weight each feature channel by how much it supports the verdict.
        verdict_score = fake_logit if p_fake >= 0.5 else -fake_logit
        gradient = torch.autograd.grad(verdict_score.sum(), feature_map)[0]
        weights = gradient.mean(dim=(2, 3), keepdim=True)
        cam = F.relu((weights * feature_map).sum(dim=1, keepdim=True))
        cam = F.interpolate(cam, size=IMG_SIZE, mode="bilinear", align_corners=False)[0, 0]
        cam = (cam / (cam.max() + 1e-8)).detach().cpu().numpy()
        result["heatmap"] = _overlay(crop, cam)
    return result


def _overlay(image, cam, strength=0.5):
    """Blend a [0, 1] heat-map over an image (blue = ignored, red = decisive)."""
    from matplotlib import colormaps
    colour = colormaps["jet"](cam)[..., :3]
    base = np.asarray(image, dtype=np.float32) / 255.0
    alpha = strength * cam[..., None]
    blended = (1 - alpha) * base + alpha * colour
    return Image.fromarray((blended * 255).astype(np.uint8)).resize((2 * IMG_SIZE, 2 * IMG_SIZE), Image.BICUBIC)

In [ ]:
%%writefile train.py
"""
train.py - reproduce the whole project with four commands:

    python train.py prepare      download Defactify and build the compact dataset in data/
    python train.py train        transfer learning + augmentation  -> models/detector.pt
    python train.py evaluate     test metrics, robustness, cost    -> results/detector/
    python train.py compare      table + chart of every run in results/

Extra runs for the report:
    python train.py train --no-aug --name no_aug
    python train.py train --model mobilenet_v3_small --name mobilenet_v3_small
    python train.py train --model efficientnet_b0 --name efficientnet_b0
"""
import argparse
import io
import json
import os
import random
import time
from collections import defaultdict
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset

from detector import (BACKBONES, IMG_SIZE, SOURCE_KEYS, SOURCES, DeepfakeDetector,
                      eval_transform, load_detector, train_transform)

ROOT = Path(__file__).resolve().parent
DATA, MODELS, RESULTS, EXAMPLES = ROOT / "data", ROOT / "models", ROOT / "results", ROOT / "examples"
SEED = 490
SOURCE_LOSS_WEIGHT = 0.3  # weight of the auxiliary "which generator" loss

# Fixed degradations used to measure robustness on the test set.
ROBUSTNESS_TESTS = {
    "clean": {},
    "JPEG 70": {"jpeg": 70},
    "JPEG 40": {"jpeg": 40},
    "blur": {"blur": 1.0},
    "half size": {"scale": 0.5},
    "feed (half size + JPEG 50)": {"scale": 0.5, "jpeg": 50},
}


def pick_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


# ============================================================================= 1. dataset
HF_REPO = "Rajarshi-Roy-research/Defactify_Image_Dataset"
HF_SHARDS = {"train": 7, "validation": 2}
# Images kept per split as (real, AI-generated); AI images are split evenly over the 5 generators.
SPLIT_SIZES = {"train": (7000, 7000), "val": (500, 500), "test": (1000, 1000)}
N_EXAMPLE_PAIRS = 6


def _save_resized(job):
    image_bytes, path, short_side = job
    img = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    scale = short_side / min(img.size)
    if scale < 1:
        img = img.resize((round(img.width * scale), round(img.height * scale)), Image.LANCZOS)
    img.save(path, "JPEG", quality=92)


def download_shards(cache):
    """Fetch the Defactify train / validation parquet files, skipping any already on disk."""
    from huggingface_hub import hf_hub_download
    shards = {}
    for split, n in HF_SHARDS.items():
        names = [f"data/{split}-{i:05d}-of-{n:05d}.parquet" for i in range(n)]
        shards[split] = [str(cache / name) if (cache / name).exists() else
                         hf_hub_download(HF_REPO, name, repo_type="dataset", local_dir=cache) for name in names]
    return shards


def prepare(args):
    """Download the public Defactify dataset and write a small balanced copy to data/."""
    import pyarrow.parquet as pq

    rng = random.Random(SEED)
    shards = download_shards(DATA / "_download")

    def read_index(files):
        """(file, row, source id, caption) for every row; source id 0 = real, 1..5 = generator."""
        rows = []
        for f in files:
            table = pq.read_table(f, columns=["Label_B", "Caption"]).to_pydict()
            rows += [(f, i, b, c) for i, (b, c) in enumerate(zip(table["Label_B"], table["Caption"]))]
        return rows

    def sample(rows, n_real, n_fake, taken=()):
        """Pick n_real real rows and n_fake AI rows (balanced over generators)."""
        by_source = defaultdict(list)
        for row in rows:
            if row[:2] not in taken:
                by_source[row[2]].append(row)
        picked = []
        for source, group in sorted(by_source.items()):
            rng.shuffle(group)
            picked += group[:n_real if source == 0 else n_fake // len(SOURCES)]
        return picked

    train_rows, heldout_rows = read_index(shards["train"]), read_index(shards["validation"])
    selection = {"train": sample(train_rows, *SPLIT_SIZES["train"])}
    selection["val"] = sample(heldout_rows, *SPLIT_SIZES["val"])
    val_keys = {row[:2] for row in selection["val"]}
    selection["test"] = sample(heldout_rows, *SPLIT_SIZES["test"], taken=val_keys)

    # Demo pairs: the same caption as a real photo and as an AI image, never seen in train / val.
    by_caption = defaultdict(dict)
    for row in heldout_rows:
        if row[:2] not in val_keys:
            by_caption[row[3]][row[2]] = row
    pairs = [(c, v) for c, v in sorted(by_caption.items()) if 0 in v and len(v) > 1]
    rng.shuffle(pairs)
    example_rows, used = {}, defaultdict(int)
    for k, (caption, versions) in enumerate(pairs[:N_EXAMPLE_PAIRS], start=1):
        source = min((s for s in versions if s != 0), key=lambda s: (used[s], s))  # cover every generator
        used[source] += 1
        fake = versions[source]
        example_rows[versions[0][:2]] = EXAMPLES / f"pair{k}_real.jpg"
        example_rows[fake[:2]] = EXAMPLES / f"pair{k}_{SOURCE_KEYS[fake[2] - 1]}.jpg"

    # Map every selected (file, row) to its output path, then extract in one pass per shard.
    targets = defaultdict(list)
    counts = {}
    for split, rows in selection.items():
        counter = defaultdict(int)
        for f, i, source, _ in rows:
            key = "real" if source == 0 else SOURCE_KEYS[source - 1]
            counter[key] += 1
            folder = DATA / split / ("real" if source == 0 else "fake")
            targets[(f, i)].append((folder / f"{key}_{counter[key]:05d}.jpg", 256))
        counts[split] = dict(counter)
    for key, path in example_rows.items():
        targets[key].append((path, 512))
    for paths in targets.values():
        for path, _ in paths:
            path.parent.mkdir(parents=True, exist_ok=True)

    done = 0
    with ThreadPoolExecutor() as pool:
        for f in shards["train"] + shards["validation"]:
            row = 0
            for batch in pq.ParquetFile(f).iter_batches(batch_size=256, columns=["Image"]):
                jobs = []
                for image in batch.column("Image").to_pylist():
                    jobs += [(image["bytes"], p, s) for p, s in targets.get((f, row), [])]
                    row += 1
                list(pool.map(_save_resized, jobs))
                done += len(jobs)
            print(f"  {Path(f).name}: {done} images written", flush=True)

    info = {"source": f"https://huggingface.co/datasets/{HF_REPO}",
            "paper": "A Comprehensive Dataset for Human vs. AI Generated Image Detection (Defactify)",
            "real_images": "MS COCO photographs", "generators": SOURCES, "seed": SEED,
            "preprocessing": "short side resized to 256 px, JPEG quality 92 (identical for both classes)",
            "images_per_split": counts}
    (DATA / "dataset_info.json").write_text(json.dumps(info, indent=2))
    print(json.dumps(counts, indent=2))
    print(f"Dataset ready in {DATA}. The raw download in {DATA / '_download'} can be deleted.")


class RealFakeFolder(Dataset):
    """data/<split>/real/*.jpg and data/<split>/fake/<generator>_*.jpg"""

    def __init__(self, split, transform):
        self.transform = transform
        self.items = [(p, 0, -1) for p in sorted((DATA / split / "real").glob("*.jpg"))]
        for p in sorted((DATA / split / "fake").glob("*.jpg")):
            self.items.append((p, 1, SOURCE_KEYS.index(p.name.split("_")[0])))
        if not self.items:
            raise SystemExit(f"No images in {DATA / split}. Run:  python train.py prepare")

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label, source = self.items[i]
        return self.transform(Image.open(path).convert("RGB")), label, source


def numpy_collate(batch):
    """Collate to NumPy arrays: batches then travel from the workers by value. (Torch's shared-memory
    tensors make every worker take 5 s to shut down on macOS.)"""
    return tuple(np.stack([np.asarray(item) for item in column]) for column in zip(*batch))


def make_loader(split, transform, args, shuffle=False, batch_size=None):
    return DataLoader(RealFakeFolder(split, transform), batch_size=batch_size or args.batch_size, shuffle=shuffle,
                      drop_last=shuffle, num_workers=args.workers, persistent_workers=args.workers > 0,
                      collate_fn=numpy_collate)


# ============================================================================= 2. training
def run_epoch(model, loader, device, optimizer=None, scheduler=None, scaler=None):
    """One pass over a loader; trains if an optimizer is given. Returns (mean loss, accuracy)."""
    training = optimizer is not None
    model.train(training)
    if training and not next(model.features.parameters()).requires_grad:
        model.features.eval()  # frozen backbone: keep its BatchNorm statistics untouched
    total_loss, correct, seen = 0.0, 0, 0
    for images, labels, sources in loader:
        images, labels, sources = [torch.from_numpy(a).to(device) for a in (images, labels, sources)]
        is_fake = labels == 1
        with torch.set_grad_enabled(training), torch.autocast("cuda", enabled=device.type == "cuda"):
            fake_logit, source_logits = model(images)
            loss = F.binary_cross_entropy_with_logits(fake_logit.float(), labels.float())
            if is_fake.any():
                loss = loss + SOURCE_LOSS_WEIGHT * F.cross_entropy(source_logits[is_fake].float(), sources[is_fake])
        if training:
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            if scheduler:
                scheduler.step()
        total_loss += loss.item() * len(labels)
        correct += ((fake_logit > 0) == is_fake).sum().item()
        seen += len(labels)
    return total_loss / seen, correct / seen


def train(args):
    set_seed()
    device = pick_device()
    name = args.name
    out_dir = RESULTS / name
    out_dir.mkdir(parents=True, exist_ok=True)
    MODELS.mkdir(exist_ok=True)
    augment = not args.no_aug
    print(f"Run '{name}': {args.model}, augmentation={'on' if augment else 'off'}, device={device}")

    train_loader = make_loader("train", train_transform(augment), args, shuffle=True)
    val_loader = make_loader("val", eval_transform(), args)
    model = DeepfakeDetector(args.model, pretrained=True).to(device)

    # Transfer learning in two phases: (1) heads only on frozen ImageNet features, (2) fine-tune everything.
    phases = [("heads only", args.head_epochs, 1e-3, True), ("fine-tune", args.epochs, args.lr, False)]
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")  # mixed precision on Colab GPUs
    history, best_acc, start = [], 0.0, time.time()
    for phase, epochs, lr, frozen in phases:
        model.freeze_backbone(frozen)
        params = [p for p in model.parameters() if p.requires_grad]
        optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=1e-4)
        scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=lr, total_steps=epochs * len(train_loader),
                                                        pct_start=0.15)
        for _ in range(epochs):
            t0 = time.time()
            train_loss, train_acc = run_epoch(model, train_loader, device, optimizer, scheduler, scaler)
            val_loss, val_acc = run_epoch(model, val_loader, device)
            history.append({"epoch": len(history) + 1, "phase": phase, "train_loss": train_loss,
                            "train_acc": train_acc, "val_loss": val_loss, "val_acc": val_acc,
                            "seconds": time.time() - t0})
            h = history[-1]
            print(f"  epoch {h['epoch']:2d} [{phase:10s}] train loss {h['train_loss']:.3f} acc {h['train_acc']:.3f} | "
                  f"val loss {h['val_loss']:.3f} acc {h['val_acc']:.3f} | {h['seconds']:.0f}s", flush=True)
            if val_acc >= best_acc:
                best_acc = val_acc
                torch.save({"state_dict": model.state_dict(), "backbone": args.model, "augment": augment,
                            "epoch": h["epoch"], "val_acc": best_acc}, MODELS / f"{name}.pt")
    minutes = (time.time() - start) / 60
    (out_dir / "history.json").write_text(json.dumps({"train_minutes": minutes, "device": str(device),
                                                       "epochs": history}, indent=2))
    print(f"Best validation accuracy {best_acc:.4f} in {minutes:.1f} min -> {MODELS / f'{name}.pt'}")
    del train_loader, val_loader
    calibrate(name, args, device)
    evaluate(args)


# ============================================================================= 3. evaluation
class RobustnessViews:
    """Applies each degradation to the same image and stacks the results: [degradations, 3, H, W]."""

    def __init__(self, degradations):
        self.views = [eval_transform(**degradation) for degradation in degradations]

    def __call__(self, img):
        return torch.stack([view(img) for view in self.views])


@torch.no_grad()
def predict(model, split, degradations, args, device):
    """Returns labels [N], generator ids [N], P(AI) under each degradation [N, degradations],
    and the predicted generator on the first view [N]."""
    loader = make_loader(split, RobustnessViews(degradations), args, batch_size=max(8, args.batch_size // 4))
    out = defaultdict(list)
    for views, labels, sources in loader:
        fake_logit, source_logits = model(torch.from_numpy(views).flatten(0, 1).to(device))
        out["p_fake"].append(torch.sigmoid(fake_logit).view(len(labels), -1).cpu().numpy())
        out["source_pred"].append(source_logits.view(len(labels), views.shape[1], -1)[:, 0].argmax(1).cpu().numpy())
        out["label"].append(labels)
        out["source"].append(sources)
    return {k: np.concatenate(v) for k, v in out.items()}


def calibrate(name, args, device):
    """Find the decision threshold that is best on the validation set, then shift the output bias of the
    saved model so that this threshold becomes 0.5 (the rest of the code can keep using 0.5)."""
    path = MODELS / f"{name}.pt"
    checkpoint = torch.load(path, map_location="cpu", weights_only=True)
    out = predict(load_detector(path, device), "val", [{}], args, device)
    p, y = out["p_fake"][:, 0], out["label"]
    grid = np.linspace(0.05, 0.95, 91)
    accuracy = np.array([((p >= t) == y).mean() for t in grid])
    best = grid[accuracy == accuracy.max()]
    threshold = float(best[np.abs(best - 0.5).argmin()])  # among equally good thresholds, the closest to 0.5
    checkpoint["state_dict"]["fake_head.bias"] -= float(np.log(threshold / (1 - threshold)))
    checkpoint["calibration_threshold"] = threshold
    torch.save(checkpoint, path)
    print(f"Calibration on the validation set: threshold {threshold:.2f} moved to 0.50 "
          f"(validation accuracy {accuracy.max():.4f})")


def measure_cost(model):
    """Parameters, weight size, GFLOPs (multiply-accumulates per image) and single-image CPU latency."""
    model = model.cpu().eval()
    x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    params = sum(p.numel() for p in model.parameters())
    macs = [0]

    def count(layer, _, out):
        if isinstance(layer, torch.nn.Conv2d):
            macs[0] += out.numel() * (layer.in_channels // layer.groups) * layer.kernel_size[0] * layer.kernel_size[1]
        else:
            macs[0] += layer.in_features * layer.out_features

    hooks = [m.register_forward_hook(count) for m in model.modules() if isinstance(m, (torch.nn.Conv2d, torch.nn.Linear))]
    with torch.no_grad():
        model(x)
        for hook in hooks:
            hook.remove()
        for _ in range(10):
            model(x)
        t0 = time.perf_counter()
        for _ in range(50):
            model(x)
    return {"params_millions": params / 1e6, "size_mb": params * 4 / 1e6, "gflops": macs[0] / 1e9,
            "cpu_ms_per_image": (time.perf_counter() - t0) / 50 * 1000}


def evaluate(args):
    from sklearn.metrics import confusion_matrix, precision_recall_fscore_support, roc_auc_score, roc_curve
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    name, device = args.name, pick_device()
    out_dir = RESULTS / name
    out_dir.mkdir(parents=True, exist_ok=True)
    checkpoint = torch.load(MODELS / f"{name}.pt", map_location="cpu", weights_only=True)
    model = load_detector(MODELS / f"{name}.pt", device)

    clean = predict(model, "test", list(ROBUSTNESS_TESTS.values()), args, device)
    y, p = clean["label"], clean["p_fake"][:, 0]  # column 0 = the clean, undegraded test
    robustness = {test: float(((clean["p_fake"][:, k] >= 0.5) == y).mean()) for k, test in enumerate(ROBUSTNESS_TESTS)}
    pred = (p >= 0.5).astype(int)
    precision, recall, f1, _ = precision_recall_fscore_support(y, pred, average="binary")
    cm = confusion_matrix(y, pred)
    fake = y == 1
    per_class = {"Real photos": float((pred[~fake] == 0).mean())}
    for i, label in enumerate(SOURCES.values()):
        per_class[label] = float(pred[clean["source"] == i].mean())

    history_file = out_dir / "history.json"
    history = json.loads(history_file.read_text()) if history_file.exists() else None
    metrics = {
        "name": name, "backbone": checkpoint["backbone"], "augmentation": checkpoint["augment"],
        "calibration_threshold": checkpoint.get("calibration_threshold"), "test_images": int(len(y)),
        "accuracy": float((pred == y).mean()), "precision": float(precision), "recall": float(recall),
        "f1": float(f1), "roc_auc": float(roc_auc_score(y, p)),
        "confusion_matrix": {"real_as_real": int(cm[0, 0]), "real_as_ai": int(cm[0, 1]),
                             "ai_as_real": int(cm[1, 0]), "ai_as_ai": int(cm[1, 1])},
        "correct_by_class": per_class,
        "generator_identification_accuracy": float((clean["source_pred"][fake] == clean["source"][fake]).mean()),
        "robustness_accuracy": robustness,
        "cost": {**measure_cost(model), "train_minutes": history["train_minutes"] if history else None,
                 "train_device": history["device"] if history else None},
    }
    (out_dir / "metrics.json").write_text(json.dumps(metrics, indent=2))

    # ---- figures
    fig, ax = plt.subplots(2, 2, figsize=(12, 9.5))
    ax[0, 0].imshow(cm, cmap="Blues")
    for (r, c), value in np.ndenumerate(cm):
        ax[0, 0].text(c, r, value, ha="center", va="center", fontsize=16,
                      color="white" if value > cm.max() / 2 else "black")
    ax[0, 0].set(xticks=[0, 1], yticks=[0, 1], xticklabels=["Real", "AI"], yticklabels=["Real", "AI"],
                 xlabel="Predicted", ylabel="Actual", title=f"Confusion matrix (accuracy {metrics['accuracy']:.1%})")
    fpr, tpr, _ = roc_curve(y, p)
    ax[0, 1].plot(fpr, tpr, lw=2)
    ax[0, 1].plot([0, 1], [0, 1], "--", color="gray", lw=1)
    ax[0, 1].set(xlabel="False positive rate", ylabel="True positive rate",
                 title=f"ROC curve (AUC {metrics['roc_auc']:.3f})")
    for axis, values, title, colour in [(ax[1, 0], per_class, "Correctly classified, by image source", "#3b7dd8"),
                                        (ax[1, 1], robustness, "Accuracy under feed-style degradation", "#2a9d8f")]:
        bars = axis.barh(list(values)[::-1], [v * 100 for v in values.values()][::-1], color=colour)
        axis.bar_label(bars, fmt="%.1f%%", padding=3)
        axis.set(xlim=(0, 112), xlabel="%", title=title)
    fig.suptitle(f"{name}: {checkpoint['backbone']}, augmentation {'on' if checkpoint['augment'] else 'off'}")
    fig.tight_layout()
    fig.savefig(out_dir / "evaluation.png", dpi=150)
    plt.close(fig)

    if history:
        epochs = history["epochs"]
        xs = [e["epoch"] for e in epochs]
        fig, ax = plt.subplots(1, 2, figsize=(11, 4))
        for axis, key, title in [(ax[0], "loss", "Loss"), (ax[1], "acc", "Accuracy")]:
            axis.plot(xs, [e[f"train_{key}"] for e in epochs], "o-", label="train")
            axis.plot(xs, [e[f"val_{key}"] for e in epochs], "o-", label="validation")
            unfreeze = sum(e["phase"] == "heads only" for e in epochs) + 0.5
            axis.axvline(unfreeze, color="gray", ls="--", lw=1)
            axis.set(xlabel="Epoch", title=f"{title}  (dashed line: backbone unfrozen)")
            axis.legend()
        fig.tight_layout()
        fig.savefig(out_dir / "training_curves.png", dpi=150)
        plt.close(fig)

    print(f"\nTest results for '{name}' ({metrics['test_images']} images)")
    print(f"  accuracy {metrics['accuracy']:.4f} | precision {precision:.4f} | recall {recall:.4f} | "
          f"F1 {f1:.4f} | AUC {metrics['roc_auc']:.4f}")
    print("  correct by class: " + ", ".join(f"{k} {v:.1%}" for k, v in per_class.items()))
    print(f"  generator identification accuracy: {metrics['generator_identification_accuracy']:.1%}")
    print("  robustness: " + ", ".join(f"{k} {v:.1%}" for k, v in robustness.items()))
    c = metrics["cost"]
    print(f"  cost: {c['params_millions']:.2f} M params, {c['size_mb']:.1f} MB, "
          f"{c['gflops']:.2f} GFLOPs, {c['cpu_ms_per_image']:.1f} ms/image on CPU")
    print(f"  saved to {out_dir}")


# ============================================================================= 4. comparison
def compare(args):
    import matplotlib
    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    runs = [json.loads(f.read_text()) for f in sorted(RESULTS.glob("*/metrics.json"))]
    if not runs:
        raise SystemExit("No results yet. Run:  python train.py train")
    feed = list(ROBUSTNESS_TESTS)[-1]
    lines = ["| Run | Backbone | Augmentation | Accuracy | AUC | Feed accuracy | Params (M) | GFLOPs | CPU ms/img | Train min |",
             "|---|---|---|---|---|---|---|---|---|---|"]
    for m in runs:
        c = m["cost"]
        lines.append(f"| {m['name']} | {m['backbone']} | {'yes' if m['augmentation'] else 'no'} | "
                     f"{m['accuracy']:.2%} | {m['roc_auc']:.4f} | {m['robustness_accuracy'][feed]:.2%} | "
                     f"{c['params_millions']:.2f} | {c['gflops']:.2f} | {c['cpu_ms_per_image']:.1f} | "
                     f"{c['train_minutes'] or float('nan'):.1f} |")
    table = "\n".join(lines)
    (RESULTS / "comparison.md").write_text(f"# Run comparison\n\n'Feed accuracy' = test accuracy after {feed}.\n\n{table}\n")
    print(table)

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.8))
    for m in runs:
        ax[0].scatter(m["cost"]["gflops"], m["accuracy"] * 100, s=90)
        ax[0].annotate(m["name"], (m["cost"]["gflops"], m["accuracy"] * 100),
                       textcoords="offset points", xytext=(7, 5), fontsize=9)
    ax[0].set(xlabel="Compute per image (GFLOPs)", ylabel="Test accuracy (%)", title="Accuracy vs computational cost")
    ax[0].margins(0.25)
    tests = list(ROBUSTNESS_TESTS)
    width = 0.8 / len(runs)
    for k, m in enumerate(runs):
        ax[1].bar(np.arange(len(tests)) + k * width, [m["robustness_accuracy"][t] * 100 for t in tests], width,
                  label=m["name"])
    ax[1].set_xticks(np.arange(len(tests)) + 0.4 - width / 2, [t.replace(" (", "\n(") for t in tests], fontsize=8)
    ax[1].set(ylabel="Test accuracy (%)", title="Robustness to feed-style degradation", ylim=(40, 112))
    ax[1].legend(fontsize=8, loc="upper center", ncol=len(runs))
    fig.tight_layout()
    fig.savefig(RESULTS / "comparison.png", dpi=150)
    print(f"Saved {RESULTS / 'comparison.md'} and {RESULTS / 'comparison.png'}")


# ============================================================================= command line
def main():
    parser = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    parser.add_argument("command", choices=["prepare", "train", "evaluate", "compare"])
    parser.add_argument("--model", default="mobilenet_v3_large", choices=list(BACKBONES))
    parser.add_argument("--name", default="detector", help="run name: models/<name>.pt and results/<name>/")
    parser.add_argument("--no-aug", action="store_true", help="train without data augmentation (ablation)")
    parser.add_argument("--head-epochs", type=int, default=2, help="epochs with the backbone frozen")
    parser.add_argument("--epochs", type=int, default=8, help="fine-tuning epochs")
    parser.add_argument("--lr", type=float, default=5e-4, help="peak fine-tuning learning rate")
    parser.add_argument("--batch-size", type=int, default=64)
    parser.add_argument("--workers", type=int, default=min(4, os.cpu_count() or 2))
    args = parser.parse_args()
    {"prepare": prepare, "train": train, "evaluate": evaluate, "compare": compare}[args.command](args)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile app.py
"""
app.py - the demo screen: place two images side by side and check which one is AI-generated.

    python app.py            open the demo at http://127.0.0.1:7860
    python app.py --share    also create a public link (use this on Colab)
"""
import argparse
import json
from pathlib import Path

import gradio as gr

from detector import analyze, load_detector

ROOT = Path(__file__).resolve().parent
MODEL_FILE = ROOT / "models" / "detector.pt"
if not MODEL_FILE.exists():
    raise SystemExit("models/detector.pt not found. Train it first:  python train.py train")
MODEL = load_detector(MODEL_FILE, device="cpu")  # a light model: the CPU is fast enough for a live demo

UNCERTAIN_BAND = 0.15  # probabilities within 50% +/- this are reported as "uncertain"

CSS = """
.verdict {border-radius: 14px; padding: 16px 18px; font-family: system-ui, sans-serif;}
.verdict, .verdict * {color: #fff !important;}
.verdict.real {background: #17804d;} .verdict.fake {background: #c0392b;} .verdict.unsure {background: #b7791f;}
.verdict .tag {font-size: 26px; font-weight: 800; letter-spacing: .5px;}
.verdict .pct {font-size: 15px; margin-top: 2px; opacity: .95;}
.verdict .bar {height: 8px; border-radius: 4px; background: rgba(255,255,255,.3); margin: 10px 0 8px;}
.verdict .bar div {height: 100%; border-radius: 4px; background: #fff;}
.verdict .meta {font-size: 14px; opacity: .95;}
.summary {text-align: center; font-size: 20px; font-weight: 700; padding: 12px; border-radius: 12px;
          border: 2px solid var(--border-color-primary);}
.empty {text-align: center; opacity: .6; padding: 22px;}
"""


def verdict_card(result):
    """HTML card for one analysed image."""
    if result is None:
        return '<div class="empty">No image</div>'
    p = result["p_fake"]
    is_fake = p >= 0.5
    if abs(p - 0.5) < UNCERTAIN_BAND:
        style, tag = "unsure", f"UNCERTAIN (leaning {'AI' if is_fake else 'real'})"
    else:
        style, tag = ("fake", "AI-GENERATED") if is_fake else ("real", "REAL PHOTO")
    meta = "No generator fingerprint found."
    if is_fake:
        generator, share = max(result["sources"].items(), key=lambda kv: kv[1])
        meta = (f"Looks most like <b>{generator}</b> ({share:.0%} match)" if share >= 0.5 else
                f"No clear generator fingerprint (closest: {generator}, {share:.0%})")
    return (f'<div class="verdict {style}"><div class="tag">{tag}</div>'
            f'<div class="pct">Probability of being AI-generated: <b>{p:.1%}</b></div>'
            f'<div class="bar"><div style="width:{p * 100:.1f}%"></div></div>'
            f'<div class="meta">{meta}</div></div>')


def summary_line(a, b):
    """One sentence comparing the two images."""
    if a is None or b is None:
        return ""
    fake_a, fake_b = a["p_fake"] >= 0.5, b["p_fake"] >= 0.5
    if fake_a and fake_b:
        text = "Both images look AI-generated."
    elif not fake_a and not fake_b:
        text = "Both images look like real photos."
    else:
        text = f"Image {'A' if fake_a else 'B'} is the AI-generated one; image {'B' if fake_a else 'A'} is the real photo."
    return f'<div class="summary">{text}</div>'


def check(image_a, image_b):
    if image_a is None and image_b is None:
        raise gr.Error("Add at least one image first.")
    a, b = [analyze(MODEL, image) if image is not None else None for image in (image_a, image_b)]
    return (summary_line(a, b), verdict_card(a), a and a["heatmap"], verdict_card(b), b and b["heatmap"])


def example_pairs():
    """examples/pairN_real.jpg + examples/pairN_<generator>.jpg, alternating which side is real."""
    pairs = []
    for k, real in enumerate(sorted((ROOT / "examples").glob("pair*_real.jpg"))):
        fake = next(p for p in real.parent.glob(real.name.split("_")[0] + "_*.jpg") if p != real)
        pairs.append([str(real), str(fake)] if k % 2 == 0 else [str(fake), str(real)])
    return pairs


def model_facts():
    """One line of test-set facts for the header, read from the last evaluation."""
    metrics_file = ROOT / "results" / "detector" / "metrics.json"
    if not metrics_file.exists():
        return ""
    m = json.loads(metrics_file.read_text())
    backbone = m["backbone"].replace("mobilenet_v", "MobileNetV").replace("efficientnet_b", "EfficientNet-B") \
        .replace("_large", "-Large").replace("_small", "-Small")
    return (f"Model: {backbone} ({m['cost']['params_millions']:.1f} M parameters, "
            f"{m['cost']['cpu_ms_per_image']:.0f} ms per image on CPU) &nbsp;|&nbsp; "
            f"test accuracy {m['accuracy']:.1%} on {m['test_images']} unseen images")


def build():
    with gr.Blocks(title="Real or AI? Deepfake image detector") as demo:
        gr.Markdown("# Real or AI?\nPlace two images side by side. The detector says which one is a real photo "
                    "and which one was AI-generated, names the generator it resembles, and shows where it looked.\n\n"
                    + model_facts())
        with gr.Row():
            image_a = gr.Image(type="pil", label="Image A", sources=["upload", "clipboard"], height=340)
            image_b = gr.Image(type="pil", label="Image B", sources=["upload", "clipboard"], height=340)
        button = gr.Button("Check both images", variant="primary", size="lg")
        summary = gr.HTML()
        with gr.Row():
            with gr.Column():
                card_a = gr.HTML()
                heat_a = gr.Image(label="Heat-map A: where the model looked", interactive=False, height=340)
            with gr.Column():
                card_b = gr.HTML()
                heat_b = gr.Image(label="Heat-map B: where the model looked", interactive=False, height=340)
        outputs = [summary, card_a, heat_a, card_b, heat_b]
        button.click(check, [image_a, image_b], outputs)
        pairs = example_pairs()
        if pairs:
            gr.Examples(pairs, [image_a, image_b], outputs, fn=check, run_on_click=True, cache_examples=False,
                        label="Example pairs from the unseen test data: the same scene as a real photo and as an AI image")
        gr.Markdown("Heat-map colours: red = regions that weighed most in the verdict, blue = ignored.  \n"
                    "Trained on real MS COCO photos and images from Stable Diffusion 2.1, SDXL, Stable Diffusion 3, "
                    "DALL-E 3 and Midjourney v6. Images from other generators, or heavily edited ones, can fool it.")
    return demo


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--share", action="store_true", help="create a public link (needed on Colab)")
    parser.add_argument("--port", type=int, default=7860)
    parser.add_argument("--no-browser", action="store_true", help="do not open a browser tab automatically")
    args = parser.parse_args()
    build().launch(share=args.share, server_port=args.port, inbrowser=not (args.share or args.no_browser),
                   css=CSS, theme=gr.themes.Soft())

## 3. Dataset
**Defactify** (*A Comprehensive Dataset for Human vs. AI Generated Image Detection*): real MS COCO photographs and
images generated from the same captions by Stable Diffusion 2.1, SDXL, Stable Diffusion 3, DALL-E 3 and Midjourney v6.

`train.py prepare` downloads the public train and validation splits (about 3.9 GB) and keeps a compact, balanced
subset: **7,000 real + 7,000 AI** images for training, **500 + 500** for validation and **1,000 + 1,000** for the
final test. Every image, real or AI, goes through the same resize (short side 256 px) and JPEG encoding, so the
model cannot cheat by reading the file format.

If you already have the submitted `data.zip`, upload it and run `!unzip -q data.zip` instead of this cell.

In [ ]:
!python train.py prepare

In [ ]:
import random
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

random.seed(0)
real = random.sample(sorted(Path("data/train/real").glob("*.jpg")), 5)
fake = [random.choice(sorted(Path("data/train/fake").glob(f"{key}_*.jpg")))
        for key in ["sd21", "sdxl", "sd3", "dalle3", "midjourney"]]
fig, axes = plt.subplots(2, 5, figsize=(17, 7))
for ax, path in zip(axes[0], real):
    ax.imshow(Image.open(path)); ax.set_title("REAL photo"); ax.axis("off")
for ax, path in zip(axes[1], fake):
    ax.imshow(Image.open(path)); ax.set_title("AI: " + path.name.split("_")[0]); ax.axis("off")
plt.tight_layout(); plt.show()

## 4. Data augmentation
Images on social media are never pristine: they are cropped, shrunk, blurred and re-compressed. A detector trained
only on clean files learns fragile cues that disappear as soon as the image is shared. `train_transform` therefore
applies, on the fly: random crop + flip, colour jitter, and the **feed augmentation** (random down-scaling, Gaussian
blur and JPEG re-compression). Below: one training image and five augmented versions of it.

In [ ]:
import torch
from detector import MEAN, STD, train_transform

image = Image.open(real[0]).convert("RGB")
augment = train_transform(augment=True)
fig, axes = plt.subplots(1, 6, figsize=(18, 3.6))
axes[0].imshow(image); axes[0].set_title("original"); axes[0].axis("off")
for ax in axes[1:]:
    x = augment(image) * torch.tensor(STD)[:, None, None] + torch.tensor(MEAN)[:, None, None]
    ax.imshow(x.permute(1, 2, 0).clamp(0, 1)); ax.set_title("augmented"); ax.axis("off")
plt.tight_layout(); plt.show()

## 5. Training (transfer learning)
The backbone is **MobileNetV3-Large pretrained on ImageNet**. Two small linear heads sit on top of it: one answers
*real or AI?*, the other answers *which generator does it look like?*.

* **Phase 1 - heads only (2 epochs):** the pretrained backbone is frozen, only the new heads learn.
* **Phase 2 - fine-tuning (8 epochs):** the whole network is unfrozen and trained with a small learning rate.

The best checkpoint is kept, its decision threshold is calibrated on the validation set, and it is then
evaluated on the test set.

In [ ]:
!python train.py train

In [ ]:
import json
from IPython.display import Image as Show, display

display(Show("results/detector/training_curves.png"))
display(Show("results/detector/evaluation.png"))
print(json.dumps(json.load(open("results/detector/metrics.json")), indent=2))

## 6. Is the augmentation worth it? Is the backbone the right one?
Three extra runs answer the two design questions of the project (optional - they take a few minutes each):

* `no_aug` - the same model trained **without** augmentation, to measure what augmentation buys in robustness
* `mobilenet_v3_small` and `efficientnet_b0` - a lighter and a heavier backbone, to pick the best
  accuracy for the lowest computational cost

In [ ]:
!python train.py train --no-aug --name no_aug
!python train.py train --model mobilenet_v3_small --name mobilenet_v3_small
!python train.py train --model efficientnet_b0 --name efficientnet_b0

In [ ]:
!python train.py compare
display(Show("results/comparison.png"))

## 7. Demo
Place two images side by side and press **Check both images**. For each image the screen shows the verdict, the
probability of being AI-generated, the generator it resembles most, and a heat-map of where the model looked.
The example pairs at the bottom come from the unseen test data.

In [ ]:
import gradio as gr
import app

app.build().launch(share=True, css=app.CSS, theme=gr.themes.Soft())